# Tugas - Perbandingan Exact NN, Annoy, HNSW, dan FAISS

**Nama:** Radit Firansah  
**NIM:** 244107020196

Menjalankan perbandingan keempat metode pada 10.000 data 128 dimensi.

In [1]:
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib
import faiss
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n_samples = 10000
d = 128
X = np.random.random((n_samples, d)).astype('float32')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10

In [2]:
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 0.707 s


In [3]:
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 0.261 s


In [4]:
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=d)
p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 0.469 s


In [5]:
start = time.time()
quantizer = faiss.IndexFlatL2(d)
index_faiss = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
_, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 0.264 s


In [6]:
print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")


=== Ringkasan Waktu (detik) ===
Exact NN : 0.707
Annoy    : 0.261
HNSW     : 0.469
FAISS    : 0.264


## Analisa

### Perbandingan Colab/Desktop vs PyDroid3 (Android)

| Aspek | Colab / Desktop | PyDroid3 (Android) |
|---|---|---|
| CPU | x86-64 multi-core, clock tinggi | ARM mobile, core terbatas |
| Library | `faiss-cpu`, `hnswlib` tersedia native | `faiss`/`hnswlib` sering gagal dipasang, Annoy lebih mudah |
| RAM | besar (belasan GB) | terbatas (3-8 GB) |
| Hasil | waktu jauh lebih cepat | waktu jauh lebih lambat, bahkan bisa gagal |

**Kesimpulan analisa:**

1. **Exact NN** paling lambat karena menghitung jarak ke semua data. Semakin besar data dan dimensi, biayanya naik linear.
2. **Annoy** dan **HNSW** jauh lebih cepat karena hanya memeriksa sebagian kandidat (approximate), dengan hasil yang mendekati exact.
3. Di **PyDroid3**, kendala utama bukan hanya CPU yang lebih lemah, tetapi juga ketersediaan library. `hnswlib` dan `faiss` sulit di-*build* di Android, sehingga biasanya hanya **Annoy** yang berhasil dijalankan. Karena itu perbandingan penuh keempat metode lebih praktis dilakukan di Colab/desktop.
4. Urutan kecepatan yang umum teramati: **HNSW ≈ FAISS IVF < Annoy < Exact NN** (tergantung parameter `ef`, `nprobe`, dan `n_trees`).
5. Waktu pada smartphone sangat bergantung pada spesifikasi perangkat (SoC, jumlah core, dan RAM). Semakin baru dan besar RAM perangkat, semakin dekat performanya ke desktop.

*Catatan: angka pada tabel ringkasan di atas adalah hasil eksekusi di lingkungan lokal (setara Colab/desktop). Untuk PyDroid3, mahasiswa perlu menjalankan notebook yang sama di aplikasi dan mencatat waktunya pada perangkat masing-masing.*